# KaroSpace Feature Distribution Calculations

This notebook recreates the tables shown in:

- `Exploration > Features > Distribution`
- `Statistics > Features > Distribution`

By default this notebook applies `RC` normalization: library-size normalization without `log1p`. Set `statistics_normalization = "LogNormalize"` or `statistics_normalized_layer = "data"` to match those KaroSpace export options.

In [58]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse


## Plain variables

In [71]:
h5ad_path = "../tests/comet_xenium_multimodal.h5ad"

assay = "protein"  # "rna" or "protein"
protein_matrix_key = "protein"
protein_feature_table_key = "protein_var"

annotation_col = "leiden_rna"
replicate_col = "sample_id"  # Used only for the pseudobulk Statistics table

rna_genes = ["ABCA1", "ACSL4", "ADAM28", "ADAM9", "ADAMTS1"]
protein_genes = ["CD1c - TRITC"]
genes = protein_genes if assay == "protein" else rna_genes
n_cells_downsample = None  # None = all cells. Use an integer to downsample.
random_seed = 7

# Distribution display values. Defaults match KaroSpace: RC from statistics_counts_layer.
statistics_counts_layer = None  # None = adata.X. Use "counts" to use adata.layers["counts"].
statistics_normalization = "RC"  # "RC" or "LogNormalize"
statistics_scale_factor = 10000.0  # Used only for RC.
statistics_normalized_layer = None  # Example: "data". If set, use this layer directly.
pseudobulk_min_cell_counts = 10  # Same as --pseudobulk-min-cell-counts. 0 disables the filter.


## Load data and select cells/features

In [72]:
adata = ad.read_h5ad(h5ad_path)

if assay == "rna":
    assay_matrix = adata.X
    assay_source = "adata.X"
    assay_layers = adata.layers
    feature_names = [str(x) for x in adata.var_names]
elif assay == "protein":
    assay_matrix = adata.obsm[protein_matrix_key]
    assay_source = f"adata.obsm[{protein_matrix_key!r}]"
    assay_layers = {protein_matrix_key: assay_matrix}
    if f"{protein_matrix_key}_arcsinh" in adata.obsm:
        assay_layers[f"{protein_matrix_key}_arcsinh"] = adata.obsm[f"{protein_matrix_key}_arcsinh"]
    protein_var = adata.uns[protein_feature_table_key]
    feature_names = [str(x) for x in protein_var.iloc[:, 0].to_numpy()]
else:
    raise ValueError('assay must be "rna" or "protein"')

def dense(x):
    return x.toarray() if sparse.issparse(x) else np.asarray(x)

def matrix_for_layer(layer_name):
    if layer_name is None or layer_name == "X":
        return assay_matrix, assay_source
    if layer_name in assay_layers:
        return assay_layers[layer_name], f"{assay} matrix {layer_name!r}"
    if layer_name in adata.layers:
        return adata.layers[layer_name], f"adata.layers[{layer_name!r}]"
    if layer_name in adata.obsm:
        return adata.obsm[layer_name], f"adata.obsm[{layer_name!r}]"
    return assay_matrix, f"{assay_source} (matrix {layer_name!r} not found)"

def take_columns(matrix, rows, cols):
    return dense(matrix[rows, :][:, cols]).astype(float, copy=False)

def library_normalized_columns(matrix, rows, cols, target_sum):
    values = take_columns(matrix, rows, cols)
    totals = np.asarray(matrix[rows, :].sum(axis=1), dtype=float).ravel()
    scale = np.divide(target_sum, totals, out=np.zeros_like(totals, dtype=float), where=totals > 0)
    return values * scale[:, None]

def distribution_columns(rows, cols):
    if statistics_normalized_layer:
        matrix, source = matrix_for_layer(statistics_normalized_layer)
        return take_columns(matrix, rows, cols), f"{source}, no extra normalization"

    matrix, source = matrix_for_layer(statistics_counts_layer)
    mode = str(statistics_normalization).strip().lower()
    if mode == "rc":
        values = library_normalized_columns(matrix, rows, cols, statistics_scale_factor)
        return values, f"RC {source}, scale_factor={statistics_scale_factor:g}, no log1p"
    if mode in {"lognormalize", "log_normalize", "log-normalize"}:
        values = library_normalized_columns(matrix, rows, cols, 10000.0)
        return np.log1p(values), f"LogNormalize {source}, target_sum=10000, log1p"
    raise ValueError('statistics_normalization must be RC or LogNormalize')

if not genes:
    genes = feature_names[:5]
genes = [str(g) for g in genes[:5]]
feature_pos = {name: i for i, name in enumerate(feature_names)}
missing = [g for g in genes if g not in feature_pos]
if missing:
    raise ValueError(f"Features not found in adata.var_names: {missing}")
cols = [feature_pos[g] for g in genes]

valid = adata.obs[annotation_col].notna().to_numpy().copy()
if replicate_col:
    valid &= adata.obs[replicate_col].notna().to_numpy()
cell_idx = np.flatnonzero(valid)
if n_cells_downsample is not None and len(cell_idx) > int(n_cells_downsample):
    rng = np.random.default_rng(random_seed)
    cell_idx = np.sort(rng.choice(cell_idx, size=int(n_cells_downsample), replace=False))

obs = adata.obs.iloc[cell_idx].copy()
labels = obs[annotation_col].astype(str)
if pd.api.types.is_categorical_dtype(obs[annotation_col]):
    categories = [str(c) for c in obs[annotation_col].cat.categories if (labels == str(c)).any()]
else:
    categories = sorted(labels.unique())

exploration_values, exploration_source = distribution_columns(cell_idx, cols)
statistics_values, statistics_source = distribution_columns(cell_idx, cols)

exploration_df = pd.DataFrame(exploration_values, index=obs.index, columns=genes)
statistics_df = pd.DataFrame(statistics_values, index=obs.index, columns=genes)

print(f"Cells used: {len(obs):,}")
print(f"Features used: {genes}")
print(f"Exploration source: {exploration_source}")
print(f"Statistics source: {statistics_source}")
print(f"Categories: {categories}")


Cells used: 46,003
Features used: ['CD1c - TRITC']
Exploration source: RC adata.obsm['protein'], scale_factor=10000, no log1p
Statistics source: RC adata.obsm['protein'], scale_factor=10000, no log1p
Categories: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13']


/var/folders/pb/p96j3wgd6t73npk9p7p6gbjh0000gp/T/ipykernel_50189/94263762.py:76: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(obs[annotation_col]):


## Exploration > Features > Distribution

HTML table columns: `Group`, `n`, `Mean`, `Median`, `% Expr`. The default sort is `Mean` descending.

In [73]:
def exploration_distribution_table(gene):
    rows = []
    for category in categories:
        values = exploration_df.loc[labels == category, gene].to_numpy(dtype=float)
        values = values[np.isfinite(values)]
        rows.append({
            "Group": category,
            "n": int(len(values)),
            "Mean": values.mean() if len(values) else np.nan,
            "Median": np.median(values) if len(values) else np.nan,
            "% Expr": 100.0 * np.mean(values > 0) if len(values) else np.nan,
        })
    return pd.DataFrame(rows).sort_values("Mean", ascending=False, na_position="last").reset_index(drop=True)

exploration_tables = {gene: exploration_distribution_table(gene) for gene in genes}
feature_to_show = genes[0]
print(f"Exploration table for {feature_to_show}")
display(exploration_tables[feature_to_show])


Exploration table for CD1c - TRITC


,Group,n,Mean,Median,% Expr
0,8,3256,254.756013,257.695381,99.969287
1,12,820,222.458023,238.017270,95.243902
2,3,1481,221.213628,233.701454,96.016205
3,11,18,214.993137,232.698991,100.000000
4,5,589,202.751563,221.149484,93.039049
5,10,2136,182.176894,182.318657,93.726592
6,6,1423,179.186638,178.237605,95.502460
7,2,1821,126.717947,70.815370,75.947282
8,0,4444,120.470718,122.123670,99.977498
9,1,6233,89.183901,30.658783,84.389540


## Statistics > Features > Distribution: Wilcoxon method

HTML table columns: `Category`, `Mean`, `Cells`. Background mean is displayed above the table.

In [74]:
def statistics_wilcoxon_table(gene):
    rows = []
    for category in categories:
        values = statistics_df.loc[labels == category, gene].to_numpy(dtype=float)
        values = values[np.isfinite(values)]
        rows.append({
            "Category": category,
            "Mean": values.mean() if len(values) else np.nan,
            "Cells": int(len(values)),
        })
    table = pd.DataFrame(rows).sort_values("Mean", ascending=False, na_position="last").reset_index(drop=True)
    return table, statistics_df[gene].mean()

statistics_wilcoxon_tables = {}
statistics_wilcoxon_background = {}
for gene in genes:
    table, background = statistics_wilcoxon_table(gene)
    statistics_wilcoxon_tables[gene] = table
    statistics_wilcoxon_background[gene] = background

print(f"Statistics/Wilcoxon table for {feature_to_show}. Background mean: {statistics_wilcoxon_background[feature_to_show]:.4f}")
display(statistics_wilcoxon_tables[feature_to_show])


Statistics/Wilcoxon table for CD1c - TRITC. Background mean: 81.2151


,Category,Mean,Cells
0,8,254.756013,3256
1,12,222.458023,820
2,3,221.213628,1481
3,11,214.993137,18
4,5,202.751563,589
5,10,182.176894,2136
6,6,179.186638,1423
7,2,126.717947,1821
8,0,120.470718,4444
9,1,89.183901,6233


## Statistics > Features > Distribution: Pseudobulk/DESeq2 method

This recreates the displayed category mean table for the Pseudobulk/DESeq2 method. DESeq2 itself uses raw-count pseudobulks for marker testing; this table uses display-scale means. The `pseudobulk_min_cell_counts` filter matches `--pseudobulk-min-cell-counts`: cells with lower total raw counts are removed before replicate-category aggregation.

In [75]:
pseudobulk_count_matrix, _ = matrix_for_layer(statistics_counts_layer)
pseudobulk_cell_totals = np.asarray(pseudobulk_count_matrix[cell_idx, :].sum(axis=1), dtype=float).ravel()
pseudobulk_cell_mask = np.isfinite(pseudobulk_cell_totals) & (pseudobulk_cell_totals >= int(pseudobulk_min_cell_counts))

def statistics_pseudobulk_table(gene):
    tmp = obs.iloc[pseudobulk_cell_mask][[replicate_col, annotation_col]].copy()
    tmp["value"] = statistics_df.iloc[pseudobulk_cell_mask][gene].to_numpy(dtype=float)
    tmp = tmp[np.isfinite(tmp["value"])]

    per_sample = (
        tmp.groupby([replicate_col, annotation_col], observed=True)["value"]
        .agg(total="sum", cells="size")
        .reset_index()
    )
    per_sample["sample_mean"] = per_sample["total"] / per_sample["cells"]

    rows = []
    for category in categories:
        sub = per_sample[per_sample[annotation_col].astype(str) == category]
        rows.append({
            "Category": category,
            "Mean": sub["sample_mean"].mean() if len(sub) else np.nan,
            "Cells": int(sub["cells"].sum()) if len(sub) else 0,
        })
    by_replicate = per_sample.groupby(replicate_col, observed=True).agg(total=("total", "sum"), cells=("cells", "sum"))
    background = (by_replicate["total"] / by_replicate["cells"]).mean()
    table = pd.DataFrame(rows).sort_values("Mean", ascending=False, na_position="last").reset_index(drop=True)
    return table, background

statistics_pseudobulk_tables = {}
statistics_pseudobulk_background = {}
for gene in genes:
    table, background = statistics_pseudobulk_table(gene)
    statistics_pseudobulk_tables[gene] = table
    statistics_pseudobulk_background[gene] = background

print(f"Statistics/Pseudobulk table for {feature_to_show}. Background mean: {statistics_pseudobulk_background[feature_to_show]:.4f}")
display(statistics_pseudobulk_tables[feature_to_show])


Statistics/Pseudobulk table for CD1c - TRITC. Background mean: 145.8291


,Category,Mean,Cells
0,11,227.949109,18
1,12,179.731907,800
2,0,172.318499,4444
3,2,168.800283,1813
4,5,167.501944,570
5,8,164.269992,3255
6,3,159.122869,1438
7,6,156.576067,1361
8,10,154.888644,2031
9,4,153.168546,12449


## All five features

In [76]:
exploration_all = pd.concat(exploration_tables, names=["Feature", "row"]).reset_index(level="Feature").reset_index(drop=True)
wilcoxon_all = pd.concat(statistics_wilcoxon_tables, names=["Feature", "row"]).reset_index(level="Feature").reset_index(drop=True)
wilcoxon_all["Background mean"] = wilcoxon_all["Feature"].map(statistics_wilcoxon_background)
pseudobulk_all = pd.concat(statistics_pseudobulk_tables, names=["Feature", "row"]).reset_index(level="Feature").reset_index(drop=True)
pseudobulk_all["Background mean"] = pseudobulk_all["Feature"].map(statistics_pseudobulk_background)

print("Exploration > Features > Distribution")
display(exploration_all)
print("Statistics > Features > Distribution: Wilcoxon")
display(wilcoxon_all)
print("Statistics > Features > Distribution: Pseudobulk/DESeq2 display means")
display(pseudobulk_all)


Exploration > Features > Distribution


,Feature,Group,n,Mean,Median,% Expr
0,CD1c - TRITC,8,3256,254.756013,257.695381,99.969287
1,CD1c - TRITC,12,820,222.458023,238.017270,95.243902
2,CD1c - TRITC,3,1481,221.213628,233.701454,96.016205
3,CD1c - TRITC,11,18,214.993137,232.698991,100.000000
4,CD1c - TRITC,5,589,202.751563,221.149484,93.039049
5,CD1c - TRITC,10,2136,182.176894,182.318657,93.726592
6,CD1c - TRITC,6,1423,179.186638,178.237605,95.502460
7,CD1c - TRITC,2,1821,126.717947,70.815370,75.947282
8,CD1c - TRITC,0,4444,120.470718,122.123670,99.977498
9,CD1c - TRITC,1,6233,89.183901,30.658783,84.389540


Statistics > Features > Distribution: Wilcoxon


,Feature,Category,Mean,Cells,Background mean
0,CD1c - TRITC,8,254.756013,3256,81.215111
1,CD1c - TRITC,12,222.458023,820,81.215111
2,CD1c - TRITC,3,221.213628,1481,81.215111
3,CD1c - TRITC,11,214.993137,18,81.215111
4,CD1c - TRITC,5,202.751563,589,81.215111
5,CD1c - TRITC,10,182.176894,2136,81.215111
6,CD1c - TRITC,6,179.186638,1423,81.215111
7,CD1c - TRITC,2,126.717947,1821,81.215111
8,CD1c - TRITC,0,120.470718,4444,81.215111
9,CD1c - TRITC,1,89.183901,6233,81.215111


Statistics > Features > Distribution: Pseudobulk/DESeq2 display means


,Feature,Category,Mean,Cells,Background mean
0,CD1c - TRITC,11,227.949109,18,145.829087
1,CD1c - TRITC,12,179.731907,800,145.829087
2,CD1c - TRITC,0,172.318499,4444,145.829087
3,CD1c - TRITC,2,168.800283,1813,145.829087
4,CD1c - TRITC,5,167.501944,570,145.829087
5,CD1c - TRITC,8,164.269992,3255,145.829087
6,CD1c - TRITC,3,159.122869,1438,145.829087
7,CD1c - TRITC,6,156.576067,1361,145.829087
8,CD1c - TRITC,10,154.888644,2031,145.829087
9,CD1c - TRITC,4,153.168546,12449,145.829087
